# Material para as decisões do grupo

Campos `c6`, corpus `corp3`, tópicos `t5`. Quatro partes:
(a) aluguéis duvidosos, (b) os temas e seus nomes, (c) trocas de OCR da limpeza, (d) conferência de uma amostra.

Este caderno reúne o material para **quatro decisões do grupo**. Em cada parte há uma tabela com uma ou duas colunas em branco: preencha **fora do caderno** (numa cópia da tabela, planilha ou ata); o caderno só mostra os dados e não grava nada. Tudo é lido do banco na hora de executar, então os números mudam se o banco mudar.

## Preparação: ler o banco
O banco devolve no máximo 1000 linhas por consulta, então lemos em blocos (paginação). Este caderno **só lê**: não grava nada no banco nem em arquivos.

In [1]:
import sys
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ / "src"))
sys.path.insert(0, str(RAIZ / "scripts"))

import json, random
from IPython.display import HTML, display
import pandas as pd
from hdb import config, corpus
from hdb.banco import BancoSupabase
from resumo_dados import nomear   # nomes sugeridos dos temas (usa hdb.topicos.nomear quando existir)

pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 220)
banco = BancoSupabase()
db = banco.cliente

def ler_tudo(consulta, bloco=1000):
    """Executa a consulta em blocos de 1000 linhas e junta o resultado."""
    linhas, inicio = [], 0
    while True:
        parte = consulta().range(inicio, inicio + bloco - 1).execute().data
        linhas += parte
        if len(parte) < bloco:
            return pd.DataFrame(linhas)
        inicio += bloco

def periodo_curto(p):
    return "sem ano" if not isinstance(p, str) else (p if p.startswith("antes") else p.split(" ")[0])
PERIODOS = ["antes de 1831", "1831–1850", "1850–1871", "1871–1888", "sem ano"]

def trecho(texto, n):
    t = " ".join(str(texto).split())
    return t if len(t) <= n else t[:n].rstrip() + "…"

def mostrar(tabela, vazio):
    """Mostra a tabela e o n; se não há linhas, escreve a frase em vez de falhar."""
    if len(tabela) == 0:
        print(vazio)
        return
    print(f"n = {len(tabela)}")
    html = tabela.reset_index(drop=True).to_html(index=False, escape=True).replace("\n", "<br>")
    display(HTML("<style>.material td, .material th {text-align: left; vertical-align: top;}</style>"
                 + html.replace('class="dataframe"', 'class="dataframe material"')))

## Parte A. Aluguéis duvidosos
Anúncios de aluguel (`aluguel_incerto`) em que o texto não diz que a pessoa é escravizada.

In [2]:
inc = ler_tudo(lambda: db.table("anuncios_completos").select("id,periodo_legal,texto,c_evidencias")
               .in_("versao", list(config.VERSOES_ANUNCIOS)).eq("c_versao", config.VERSAO_CAMPOS)
               .eq("c_tipo", "aluguel_incerto").order("id"))
if len(inc) == 0:
    print("Não há anúncios de aluguel duvidoso nesta versão de campos.")
else:
    inc["periodo"] = pd.Categorical(inc["periodo_legal"].map(periodo_curto), categories=PERIODOS, ordered=True)
    print(f"{len(inc)} anúncios de aluguel duvidoso (campos {config.VERSAO_CAMPOS}).")
    if len(inc) != 197:
        print(f"ATENÇÃO: os comentários deste caderno foram escritos para 197 anúncios e agora são {len(inc)}: "
              "confira os números dos textos antes de usar.")
    por_periodo = inc.groupby("periodo", observed=True).size().rename("Anúncios").to_frame()
    por_periodo.index.name = "Período"
    print("\nPor período legal (n por linha):")
    display(por_periodo)

197 anúncios de aluguel duvidoso (campos c6).

Por período legal (n por linha):


,Anúncios
Período,
antes de 1831,1
1831–1850,17
1850–1871,90
1871–1888,89


In [3]:
def motivo(ev):
    if isinstance(ev, str):
        try:
            ev = json.loads(ev)
        except ValueError:
            return ev
    return (ev or {}).get("tipo", "(sem motivo registrado)")

if len(inc):
    amostra = inc.sample(min(20, len(inc)), random_state=42).sort_values("id")
    ex = pd.DataFrame({"id": amostra["id"], "Período": amostra["periodo"].astype(str),
                       "Texto (até 220 caracteres)": amostra["texto"].map(lambda t: trecho(t, 220)),
                       "Motivo (evidência do campo)": amostra["c_evidencias"].map(motivo)})
    print("20 exemplos sorteados (semente 42):")
    mostrar(ex, "Sem exemplos.")

20 exemplos sorteados (semente 42):
n = 20


id,Período,Texto (até 220 caracteres),Motivo (evidência do campo)
9932,1850–1871,"LUGA-SE uma nogrinhs com 16 ansos de idade, A pera ama do crianção, 6 levar congomaas é roupa das mesmas é ooor na rua do Principe dos Cajueiros",pessoa sem marca de escravizado: ama
10756,1871–1888,"RECISA-SE de uma criada Hi nos, casa de pequena traventa e 8 Francisco de Pa “de costureiras € uia maclinisia; né Ouvidor mn. 4. Li de ur pequeno de lã a 18 aunos de idade, para um açougue; na rua do Engeuho-Velho , ou",pessoa sem marca de escravizado: criada
10767,1871–1888,"RECISA-SE, ne ras dz Oliado nº 0, do use Udo P * ingomuadeira, de bos ccndusta livro À (osECISA-SE de hqs “oficiças de” Prstaltas os olficina, divijase é 135, cobrado. ds pra ; FUI, um menino de 12 a 14 anuos de freloese…",pessoa sem marca de escravizado: menino
10843,1871–1888,"PRECISA-SE. para casa de Pequena familia, de uma rapariga que lave é engommes para tratar, a rua da Re fandega n. 28, andar, ás rar ESEECISA-HE duma ando Jo lodo ps ECISA-SE de uma criada de meia idade, para É casa de po…",pessoa sem marca de escravizado: rapariga
13028,1871–1888,"RECISA-SE de uma criada que cozinhe, para casa P Josquim » 132. na rua do Espirito-Santo de pequena familia; ENDE! perita eogom- Videira de toda a roupa e o mais serviço de",pessoa sem marca de escravizado: criada
13676,1871–1888,"RECISA-SE de uma perfei. a engommadeira, escrava ou livre; na rua da Quitanda",pessoa sem marca de escravizado: escrava ou livre
18468,1871–1888,"LUGA-SE uma criado na rua da Conceição vadeira é cozinha, por 3/8 na ALisase ama jarda tez S. Pedro n. 166. serviço de cas: ar e corinhar, de",pessoa sem marca de escravizado: criado
18587,1871–1888,"PRECISA-SE de uma mucama para iratar do que teu aluguel não exceda de 125; na n. 121, 3º andar. lavad CISA-SE de uma cilada, raso À)35 ; na aa do Hogicio nm 260 mn Aa py ueno, com pratica de ve der doce , 2 aonos de idad…",pessoa sem marca de escravizado: mucama
18616,1871–1888,"RECISA-SE de ama repariga que saiba cosiabar, para casa de pouca familia; na rua do Oavidor a. ti para” lavar, cozinhar RE je ma pese e engommar para um casal, prefere-se de idade ; na rua do Ouvidor n. 1 loja de chapéos",pessoa sem marca de escravizado: ama
29638,1831–1850,"PRECISA-SE, no hôtel Ravot, de um criado que dê fiador à sua conducta, para o serviço dos quartos. — O ABAIXO assignado declara que, tendo con= tractado perants a autoridade legal os serviços de João Manoel Inrocencio +…",pessoa sem marca de escravizado: criado


**O que decidir:** o que fazer com os anúncios de aluguel em que o texto **não traz marca de pessoa escravizada** (`aluguel_incerto`): "aluga-se uma criada", "uma rapariga" etc. Podem ser pessoas escravizadas, forras ou livres.

**As 3 opções**

- **(a) Deixar fora** do corpus e dos tópicos (é o que fazemos hoje, variante `t5`).
- **(b) Incluir com marca**: entram no corpus e nas análises, sempre sinalizados como "incertos" (variante `t5i`).
- **(c) Incluir só com pista**: entram apenas os que têm alguma pista de condição (a evidência do campo diz qual).

**Recomendação: (b).** Já foi testada na variante `t5i`: **a direção do resultado não muda** (ver a seção de comparação no caderno `topicos_por_periodo`). Incluir com marca deixa o grupo ver os dois cenários sem perder esses anúncios.

## Parte B. Os temas
Para o modelo de tópicos da versão `t5`: nome sugerido, as 12 palavras de maior peso, quantos anúncios
têm aquele tema como o mais forte (n), de que tipos são e 5 anúncios típicos (os de maior proporção do tema).

In [4]:
MODELO = config.VERSAO_TOPICOS
pal = ler_tudo(lambda: db.table("topico_palavras").select("topico,palavra,peso").eq("versao", MODELO)
               .order("topico").order("peso", desc=True))
dist = ler_tudo(lambda: db.table("anuncio_topicos").select("anuncio_id,topico,proporcao").eq("versao", MODELO)
                .order("anuncio_id").order("topico"))
if len(pal) == 0 or len(dist) == 0:
    print(f"O modelo {MODELO} não tem dados no banco.")
else:
    ids = sorted(dist["anuncio_id"].unique())
    anu = ler_tudo(lambda: db.table("anuncios_completos").select("id,periodo_legal,c_tipo,texto")
                   .in_("versao", list(config.VERSOES_ANUNCIOS)).eq("c_versao", config.VERSAO_CAMPOS).order("id"))
    anu = anu[anu["id"].isin(ids)].drop_duplicates("id").set_index("id")
    palavras = {int(t): list(g["palavra"]) for t, g in pal.groupby("topico")}
    NOMES = nomear(palavras)
    pivo = dist.pivot(index="anuncio_id", columns="topico", values="proporcao").fillna(0)
    base = anu.join(pivo, how="inner")
    base["dominante"] = base[list(pivo.columns)].idxmax(axis=1)
    linhas = []
    for t in sorted(palavras):
        do_tema = base[base["dominante"] == t]
        tipos = do_tema["c_tipo"].value_counts(normalize=True)
        dist_tipo = ", ".join(f"{k} {100 * v:.0f}%" for k, v in tipos.items()) or "–"
        tip = base.sort_values(t, ascending=False).head(config.TOPICOS_TIPICOS)
        tipicos = "\n".join(f"• [id {i}, {r['c_tipo']}, {r[t]:.2f}] {trecho(r['texto'], 300)}" for i, r in tip.iterrows())
        linhas.append({"Tópico": f"T{t}", "Nome sugerido": NOMES[t], "12 palavras": ", ".join(palavras[t][:12]),
                       "n": len(do_tema), "Tipos (% dos anúncios do tema)": dist_tipo,
                       f"{config.TOPICOS_TIPICOS} anúncios típicos (300 caracteres)": tipicos,
                       "Nome escolhido pelo grupo": ""})
    print(f"{len(base)} anúncios no modelo {MODELO}; n = anúncios cujo tema mais forte é aquele.")
    mostrar(pd.DataFrame(linhas), "Sem temas.")

2191 anúncios no modelo t5; n = anúncios cujo tema mais forte é aquele.
n = 6


Tópico,Nome sugerido,12 palavras,n,Tipos (% dos anúncios do tema),5 anúncios típicos (300 caracteres),Nome escolhido pelo grupo
T0,Aluguel de pretos e moleques para todo o serviço,"aluga, preto, servico, todo, pretas, precisa, duas, alugao, moleque, bom, alugar, dois",326,"aluguel 74%, venda 19%, ama_de_leite 4%, fuga 1%, compra 1%","• [id 6351, venda, 0.98] VENDE-SE hum preto bom cozinheiro, na rua do Hospicio n. 24, — A. TAVEL c A. F. ZIESE fazem sciento a esta praça que, na data de koje, dissolvêrão amigavelmente a sociedade que tinhão nella com a firma de TAVEL E ZIESE, ficando encarregado da liquidação da mesma o socio A, F. ZIESE, que, com a firma…\n• [id 31291, aluguel, 0.96] LUGA-SE um moleque copeiro, e tambem se aluga da frente de um sobrado outro para todo o serviço; na rua do Cano n. bo. mens; para iaformações, = É uma pesto quo lava, cagomena 6 fas com ""OS &. 35. 1/2 wma esa qu va ngomnas fas co Pd ção todo o serviço de uma casa ; trata-se ua IGA-SE ER rua do Mowpi…\n• [id 33735, venda, 0.96] VENDEM-SE infallivelmente a quem mais der, por estar para retirar se brevemente uma familia, algumas pretas prendadas, negrinhas que ja podem fazer algum serviço, é um preto bom official de calufate quem os pretender dirija-se à rua Nova de, Fran. cisco da Prainha 0,27, primeiro andar,\n• [id 6139, venda, 0.96] VENDE-SE hum preto bom cozinheiro, com a condição de ir para o Rio Grande ou Maranhão; na Tua do Ouvidor , n. 29, — JOSÉ Manoel de Barros comprou , de sociedade com Francisco Xavier de Souza Passos, 1 meio bilhete da 7º loteria do theatro de S. Pedro d'Alcantara, n, 2140 , o qual fica em poder do an…\n• [id 36651, aluguel, 0.95] preto velho: na gua da Misericordia n. 18, loja. pesa um moço de 25 au Dos, Tonto. Piana iu de Pat, qe ano elão: z E ELISA SE de vs cosnbeio para casa dS PAR; Nua rua do Mercadon. 1. — IDRECISA-SE de uma chacarinha, com ERA casa É agua 1», Enge q sm a exoeda menaas o tiver deiso caia no ecrptorio de…",
T1,"Fuga: nome, idade e dono","dia, nome, corrente, preto, fugiu, anos, idade, nacao, escravo, sera, camisa, vestido",153,"fuga 86%, aluguel 7%, venda 5%, ama_de_leite 1%, compra 1%","• [id 6263, fuga, 0.99] — PAGAR-SE-HA O trabalho de quem pegar, o levar à rua do Conde n. 55, hum escravo de no-: me Vicente, de 24 annos de idade pardo escuro , com cabello de preto, secco, de mediana estatura ,e com pouca barba , que desappareceu hontem 16 do corrente , indo vestido com calças e camisa de riscado azul, —…\n• [id 14286, fuga, 0.98] PUBLICAÇÕES A PEDIDO. OS PAQUETES DE VAPOR. A vista do que vem publicado no Liberal n. 121 de 9 do corta é incontestas el que ninguem poderá gera fo sia reg nh Sra pancetino José Coelho, Ke: propriedade, nem com as leis e regulamentos, tanto policiaes como «la capitania do porto, vai mandando a seu…\n• [id 29624, fuga, 0.98] FUGIO, no diafs de abril de 1843, um preto de nome Fortunato, de nação Moçambique, idade 28 a 30 annos, com os signaes seguintes : tem os dedos da mão direita encolhidos, procedido de cicatrizes de fo- O, estatura regular, cara alguma cousa magra e fulà com umaorelha furada por ter usado de brinco,…\n• [id 13781, fuga, 0.98] Escravos FuaiDos . o “= 9] No dia 29 de Setembro pelas 3 horas da tarde desemcaminhou-se bum preto por nome João Nação Moçambique, com 08 ginaes: seguintes : hum sinal redondo na testa, e o mesmo nas fontes,: € com a orelha esquerda furada, é huma pequena ferida no tornozello esquerdo , com ealças.d…\n• [id 30678, fuga, 0.98] UGIO no dia 28 de Fevereiro o escravo pardo de nome Marcolino, levando vestido camiss branca de morim e calça arul, é tem os signaes segnintes : rosto redondo, plysioncmia de Chim, tem uma cicatriz ua testa, dentes inuito claros, uma orelha furada em que traz uma argola de linha preta : esto escravo…",
T2,Venda com preço e qualidades,"vende, preto, anos, bom, cozinheiro, aluga, boa, perfeito, preta, forno, moleque, escravos",439,"venda 67%, aluguel 27%, compra 3%, ama_de_leite 3%, fuga 0%","• [id 6146, vend

**O que decidir:** o nome de cada tema. Os nomes sugeridos vêm das palavras de cada tópico e são **hipóteses**. Leia as 12 palavras e os 5 anúncios típicos e preencha a coluna **Nome escolhido pelo grupo**. Lembre que os temas acompanham o tipo do anúncio (fuga, venda, aluguel, ama de leite): o nome pode seguir o tipo.

## Parte C. Trocas de OCR da limpeza
A limpeza descobre erros de OCR (palavra rara a uma letra de uma palavra muito mais frequente). Aqui repetimos a
descoberta sobre os anúncios selecionados do corpus, como em `corpus.processar`, **sem gravar nada**.

In [5]:
todos = banco.anuncios_para_corpus(config.VERSAO_CAMPOS)
sel = [a for a in todos if corpus.selecionar(a)]
if not sel:
    print("Nenhum anúncio selecionado para o corpus.")
else:
    padrao = [corpus.padronizar(a["texto"]) for a in sel]
    desc = corpus.descobrir_ocr(padrao)
    freq = {}
    for t in padrao:
        for p in set(t.split()):
            freq[p] = freq.get(p, 0) + 1
    print(f"{len(sel)} anúncios selecionados; {len(desc['aplicadas'])} trocas aplicadas, {len(desc['recusadas'])} recusadas.")
    ap = pd.DataFrame([{"Erro": w, "Correta": v, "Anúncios com o erro": freq.get(w, 0),
                        "Anúncios com a correta": freq.get(v, 0), "Aprovada? (s/n)": ""}
                       for w, v in desc["aplicadas"].items()]).sort_values("Anúncios com o erro", ascending=False) \
        if desc["aplicadas"] else pd.DataFrame()
    print("\nTrocas aplicadas:")
    mostrar(ap, "Nenhuma troca foi aplicada.")
    rec = pd.DataFrame([{"Palavra": w, "Candidata": v, "Anúncios com a palavra": n_w, "Anúncios com a candidata": n_v,
                         "Por que foi recusada": m} for w, v, n_w, n_v, m in desc["recusadas"]])
    if len(rec):
        rec = rec.sort_values("Anúncios com a palavra", ascending=False).head(10)
    print("\nAs 10 recusadas mais frequentes:")
    mostrar(rec, "Nenhuma troca foi recusada.")

2191 anúncios selecionados; 42 trocas aplicadas, 2720 recusadas.

Trocas aplicadas:
n = 42


Erro,Correta,Anúncios com o erro,Anúncios com a correta,Aprovada? (s/n)
obrado,sobrado,3,121,
ngoma,engoma,3,122,
rente,frente,3,40,
ropria,propria,2,40,
entes,dentes,2,54,
arpinteiro,carpinteiro,1,25,
abelo,cabelo,1,24,
amisa,camisa,1,73,
azenda,fazenda,1,49,
abeca,cabeca,1,24,



As 10 recusadas mais frequentes:
n = 10


Palavra,Candidata,Anúncios com a palavra,Anúncios com a candidata,Por que foi recusada
alugar,aluga,96,786,frequente
pretas,preta,83,706,frequente
pretos,preto,66,542,frequente
vendem,vende,65,522,frequente
alugao,aluga,51,786,frequente
preco,preto,50,542,frequente
venda,vende,41,522,frequente
vender,vende,41,522,frequente
tudo,todo,33,277,frequente
precisao,precisa,18,163,nao_sistematico


**O que decidir:** quais trocas de OCR a limpeza pode aplicar. O programa troca uma palavra rara por outra muito mais frequente a uma letra de distância, **só quando a rara é a outra sem a 1ª letra** (o erro típico: `luga` por `aluga`). Marque **s** nas trocas que fazem sentido e **n** nas que juntam palavras diferentes. As recusadas são as maiores que **não** foram aplicadas; se alguma deveria ser, diga.

## Parte D. Conferência de uma amostra
**Os rótulos "Claude" são de Claude, não de pessoas.** O grupo preenche as duas últimas colunas.

In [6]:
FIX = RAIZ / "tests" / "fixtures"
def carregar(nome, n=None):
    arq = FIX / nome
    if not arq.exists():
        return []
    itens = json.load(open(arq, encoding="utf-8"))
    return random.Random(42).sample(itens, min(n, len(itens))) if n else itens

def tabela_conferencia(itens):
    return pd.DataFrame([{"id": i["id"], "Período": i.get("periodo"), "Texto (até 300 caracteres)": trecho(i["texto"], 300),
                          "Tipo do código antigo (c4)": i.get("tipo_c4"), "Tipo certo (Claude)": i.get("tipo_certo"),
                          "É anúncio? (Claude)": "sim" if i.get("e_anuncio") else "não",
                          "Tipo certo segundo o grupo": "", "É anúncio de pessoa escravizada? s/n": ""} for i in itens])

print("D1. Os 40 anúncios de conferencia_extra.json")
mostrar(tabela_conferencia(carregar("conferencia_extra.json")), "Arquivo conferencia_extra.json ausente ou vazio.")

D1. Os 40 anúncios de conferencia_extra.json
n = 40


id,Período,Texto (até 300 caracteres),Tipo do código antigo (c4),Tipo certo (Claude),É anúncio? (Claude),Tipo certo segundo o grupo,É anúncio de pessoa escravizada? s/n
7763,1850–1871,"mudar de roupa quando anda fugido, sabe ler e escrever, e talvez se intitule 3 grai quem o apprehender elevará rua dos Benedictinos n. 1; as e fra quem lhe der conto. 2! OM er",fuga,fuga,sim,,
8883,1871–1888,"esarPARECEU da rus do Barão de Itaparipe 14 de Dezembro, uma besta rozilha à pés, tem um joelho enchado; oticias della, paga-se as despezas.",fuga,outro,não,,
13926,antes de 1831,"ma, e sem camisa, quando anda pucha de ham - quarto; quem o apanhar, ou der noticia delle” participando na dita casa receberá alviçaras. ô2 No Domingo de noute 6 do corrente,",fuga,fuga,sim,,
9699,1850–1871,"Desapparecen no dia 15 do corzente, da rua do Peineipo dos Cajueirca D. 159 À, um cachorro pequeno fel- » todo pelo time de Jasmim; quem o á casa seirna ser: grutificado. de",fuga,outro,não,,
13228,1871–1888,"Desoppareceu na quarta-feira, 7 do corrente E rente mez sificaçõe: uma senhora alle gir a assignatura inventor, à do uma filhinha de tres a quatro anno: do onde foi, pede-se aos Srs. proprietari Pharmecia boteis e hospedarias ou pessoas que tiverão conhecimento della o favor de participar na rua da…",fuga,outro,não,,
36026,1850–1871,"fugão, de Los conducta; no campo da Acclamação",fuga,NaN,não,,
34057,1831–1850,"DESAPPARECEU, no dia 3 de fevereiro do correste voso de arsenal do exercito, um aprendiz menor, de nome Francisco Augusto, idade de 14 anvos, natural ce Pernambuco , côr um pouco morena, Lastantes cardos pela cara etem noslto ds cabeça alguma falta de cabello procedida de um tumor que teve he *empo;…",fuga,outro,não,,
19282,1871–1888,"CAPIVARA PRAÇA DO RIO DE JANEIRO FUGIDO iro & C., estabelecidos em Pindamonhangaba, ""O + oa a qu em é Fevereiro do 1854, da fazsada : ça e ego Antonio, natural da Pa- Tm por escripto, ns o, eheio de corpo, À soci Pindamonhangaba, 23 os, bem esperto, de 1885. igeir cicatriz de do pescoço, lado es-",fuga,fuga,sim,,
17437,1871–1888,"individuos desconhecidos na praça. “e faz leilões de moseis de sua casa e de diversas partidas de lonas, ete., de seus estabelecimentos, em nome de terceir Com prejuizos, consideraveis, demonstrando linuidação precipitais, e ainda prejudicando os d Seg hroprio filhinho, fazendo desapparecer bens que…",fuga,outro,não,,
8116,1850–1871,"Fagio no dia 10 do proximo prssad: dio. ta, esorava do Antonio Albino de Souza, me: Torta: peotasta-ab contra que tiver a pe",fuga,fuga,sim,,


In [7]:
print("D2. 60 anúncios de conferencia.json (sorteados, semente 42)")
mostrar(tabela_conferencia(carregar("conferencia.json", 60)), "Arquivo conferencia.json ausente ou vazio.")

D2. 60 anúncios de conferencia.json (sorteados, semente 42)
n = 60


id,Período,Texto (até 300 caracteres),Tipo do código antigo (c4),Tipo certo (Claude),É anúncio? (Claude),Tipo certo segundo o grupo,É anúncio de pessoa escravizada? s/n
13069,1871–1888,"ENDE-SE por 8008 uma preta de meia idade, Dlias “e “uma porta, por 23009 cada um : a; na rua da Alfao- Vperteta cozinheira e lavado trata-se na rua da Candel",venda,venda,sim,,
34068,1831–1850,"FUGIO, da rua do Fogo n. 93, n. dia 20 de muio do corceste anno ums pela de neme Josefa, de nação Moçambique, de 17 a 18 annos, com os pés alguma covsa mal feitos; levou vestido de riscadinho azul lenço encarnado ao pescoço, e tem a falla alguma cousa descansada. Quem della der noticias, ou a levar…",fuga,fuga,sim,,
6120,1831–1850,"DESAPPARECEU, no dia 13 de janeiro do cor« rente anno + hum moleque de nação Congo , “de nome Ludogero, dos que forão aprezados vindos nó bri- 10 gre-escuna Curolina , o qual moleque eu entróguo à Francisco das Chagas Cunha, em 23 de abril dé 1839: levou vestido calças de riscado escuro., camisa de…",fuga,fuga,sim,,
13999,antes de 1831,"45 Domingo 13-do corrente dezappareceu hum muleque de: Nação Benguella, por nome Antomio, de idade pde ll a 12 annos, ainda meio bucal, vestido com camiza de pano de linho, calça de pano azul, einda com algumas sarnas quem delic souber dírija se a rua da Valla no sobrado N. 118, que receberá boas al…",fuga,fuga,sim,,
27116,1850–1871,"va DE set Fui anto-hontesm recolhido de guarda do morro de Panla Mattos, o es- Const rios furancio, como fugido e autor de",fuga,NaN,não,,
23051,1850–1871,"FUGIO ca botica do largo de Santa Rita n. 20, a pon Alfasema ; gratilica-so a quem o apre- 0 SOLO ADO bo MINDELLO. J or nal libers l.",fuga,NaN,não,,
29854,1831–1850,"VENDE-DE um despolpador de café, cujo machinismo é o mais simples possível; wi rua Nova d:s Bento n. 47. — VENDE SE ou aluga se uma preta ama de À com as prendas seguintes : lava, engomma, cose, cogli vestidos € caseisas de jo mem, caseia e é perfeita mucama: aliança se não ter vícios nens molestia,…",venda,venda,sim,,
34219,1831–1850,"ALUGÃO-8E dous pretos bons cozinheiros, e um moleque para pagem ; na rua do Hospício n. 430.",aluguel,aluguel,sim,,
13866,antes de 1831,"“12 Vende-se hum preto sem vicio, ou Marhas, o qual anda ganhango peias ruas, € da todos os dias 240; quem o pertender , dirija. se a rua do Alecrim da Valla para cima N. 55.º j3 Vende se buma preta muito rapariga, de idade 14 a 15 amos, muito sadia, bonita àa, lava, e cosinha-com perfeição, € arrân…",venda,venda,sim,,
6528,1831–1850,"PROCLAMAÇÃO DO NOVO PRESIDENTE, eua t ni hises !— Acaho pelo der 1 ma presidencia e cc ttuto-me, pois, comvyosco, por achar-me à te: hum povo leal, brioso e denodado e por me caber a gloria de ainft huma vez servir ao Erazil e ao throno excelso de o Senhor D. Pedro, renses ! o gento do mal, que não…",fuga,outro,não,,


**O que decidir:** se o nosso código acerta o tipo e se o trecho é anúncio de pessoa escravizada. **Atenção: os rótulos das colunas "Claude" são de Claude (um modelo de linguagem), não de pessoas; podem estar errados.** As duas últimas colunas ficam em branco para o grupo. Onde o grupo discordar, vale a resposta do grupo.